# 02 · Data Cleaning & Wrangling

Turns the raw scraped CSVs into clean, standardized tables ready for the Power BI model.

Key tasks:
1. Standardize **club names** across seasons
2. Normalize **season keys** to `2010\u201311`
3. Parse messy **transfer fees** (`\u00a389m`, `\u20ac85 million`, `\u00a317,000,000`) into numeric GBP
4. Drop Wikipedia **"Total" summary rows**


## 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import re, os

RAW_DIR = os.path.join('data', 'raw')
PROC_DIR = os.path.join('data', 'processed')
os.makedirs(PROC_DIR, exist_ok=True)


## 2. Standardize club names

Clubs appear with different spellings across seasons (abbreviations, sponsors, renames).
A mapping dictionary collapses them to one canonical name.


In [ ]:
CLUB_NAME_MAP = {
    'Man Utd': 'Manchester United', 'Man United': 'Manchester United',
    'Man City': 'Manchester City',
    'Spurs': 'Tottenham Hotspur',
    'Wolves': 'Wolverhampton Wanderers',
    'West Brom': 'West Bromwich Albion', 'West Bromwich': 'West Bromwich Albion',
    'Brighton': 'Brighton & Hove Albion',
    'Newcastle': 'Newcastle United',
    # ... extend as needed
}

def clean_club(name):
    if pd.isna(name):
        return name
    n = str(name).strip()
    n = re.sub(r'\s*\((C|R|P|O)\)\s*$', '', n)  # drop (C)=champion, (R)=relegated tags
    return CLUB_NAME_MAP.get(n, n)


## 3. Normalize season keys

Every source must use the same season label: `2010\u201311` (en-dash).


In [ ]:
def normalize_season(s):
    """Accepts '2010-2011', '2010/11', '2010-11' -> returns '2010\u201311'."""
    if pd.isna(s):
        return s
    digits = re.findall(r'\d{4}', str(s))
    if not digits:
        return s
    start = digits[0]
    end2 = str(int(start) + 1)[-2:]
    return f'{start}\u2013{end2}'  # en-dash

for raw in ['2010-2011', '2010/11', '2010-11']:
    print(raw, '->', normalize_season(raw))


## 4. Parse transfer fees → numeric GBP

Fees are free text: `\u00a317,000,000`, `\u00a389m`, `\u20ac85 million`, `Undisclosed`, `Free`.
This mirrors the DAX logic used in Power BI (kept here so the cleaned data is usable anywhere).

> EUR is converted to GBP at an approximate fixed rate (~0.85).


In [ ]:
EUR_TO_GBP = 0.85

def parse_fee(text):
    if pd.isna(text):
        return np.nan
    s = str(text).strip()
    cur = '\u00a3' if s.startswith('\u00a3') else ('\u20ac' if s.startswith('\u20ac') else None)
    if cur is None:
        return np.nan  # Undisclosed / Free / Loan / etc.
    has_million = bool(re.search(r'm|million', s, flags=re.I))
    num = re.sub(r'[^0-9.]', '', s.split('(')[0])  # keep digits and dot
    if num in ('', '.'):
        return np.nan
    val = float(num)
    if has_million and val < 10000:
        val *= 1_000_000
    if cur == '\u20ac':
        val *= EUR_TO_GBP
    return val

for f in ['\u00a317,000,000', '\u00a389m', '\u20ac85 million', 'Undisclosed', 'Free']:
    print(f'{f:20s} -> {parse_fee(f)}')


## 5. Drop Wikipedia "Total" summary rows

Transfer tables end with a `Total` / `Total transfer spending` row that must be excluded
from player-level analysis.


In [ ]:
def is_real_transfer(player):
    if pd.isna(player):
        return False
    return not str(player).strip().lower().startswith('total')


## 6. Example — clean a transfers table end-to-end


In [ ]:
# df = pd.read_csv(os.path.join(RAW_DIR, '2016-17__table_XX.csv'))  # pick the transfers table
# df['Season']    = normalize_season('2016-17')
# df['Team']      = df['Team'].map(clean_club)
# df['Fee Value'] = df['Fee'].map(parse_fee)
# df['Is Real']   = df['Player'].map(is_real_transfer)
# df = df[df['Is Real']]
# df.to_csv(os.path.join(PROC_DIR, 'transfers_in.csv'), index=False)
print('Apply the helpers above to each raw table, then save into data/processed/.')


## 7. Output

Clean tables land in `data/processed/` with consistent `Season`, `Club`, and numeric fee columns —
ready to load into Power BI and connect via the star-schema model.
